In [1]:
# ============================================
# DEEPFAKE DETECTION - 3-DATASET T=16 TRAINING
# ============================================
# Datasets: DFDC02 + DFD01 + CelebDF (all preprocessed with T=16)
# 4 ablation experiments: full, spatial, temporal, sequential
# GPU: P100 recommended (16GB)
#
# Required Kaggle inputs:
#   - preprocessed-dfdc02-16
#   - preprocessed-dfd01-16
#   - celebdf-preprocessed-t16
#
# RESUME support: if a previous run saved partial results as a Kaggle dataset
# (e.g. "3ds-t16-partial"), attach it as input and completed experiments
# will be skipped automatically.

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc, shutil
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f'numpy version: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# =============================================
# RESUME: restore partial results from previous run
# =============================================
PARTIAL_RESULTS_NAME = '3ds-t16-partial'

def restore_partial_results():
    """Search /kaggle/input for partial results from a previous run and restore them."""
    for root, dirs, files in os.walk('/kaggle/input'):
        if PARTIAL_RESULTS_NAME in os.path.basename(root).lower().replace('_', '-'):
            exp_src = None
            splits_src = None
            for d in dirs:
                if d == 'experiments':
                    exp_src = os.path.join(root, d)
                elif d == 'splits':
                    splits_src = os.path.join(root, d)
            for sub in dirs:
                subpath = os.path.join(root, sub)
                if os.path.isdir(subpath):
                    for sd in os.listdir(subpath):
                        if sd == 'experiments':
                            exp_src = os.path.join(subpath, sd)
                        elif sd == 'splits':
                            splits_src = os.path.join(subpath, sd)

            if exp_src and os.path.isdir(exp_src):
                print(f'[RESUME] Found partial results at {exp_src}')
                dst = './experiments'
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(exp_src, dst)
                exp_dirs = [d for d in os.listdir(dst) if os.path.isdir(os.path.join(dst, d))]
                print(f'[RESUME] Restored {len(exp_dirs)} experiment dirs: {exp_dirs}')

                if splits_src and os.path.isdir(splits_src):
                    sdst = './splits'
                    if os.path.exists(sdst):
                        shutil.rmtree(sdst)
                    shutil.copytree(splits_src, sdst)
                    # Also copy nested splits/ if present
                    nested = os.path.join(sdst, 'splits')
                    if os.path.isdir(nested):
                        for f in os.listdir(nested):
                            shutil.copy2(os.path.join(nested, f), os.path.join(sdst, f))
                    print(f'[RESUME] Restored splits from {splits_src}')

                return True
    return False

restored = restore_partial_results()
if not restored:
    print('[RESUME] No partial results found — starting fresh')

# =============================================
# Find ALL T=16 datasets in /kaggle/input
# =============================================
TARGET_T = 16
print(f'\n=== Searching for T={TARGET_T} datasets ===')
found_datasets = []
for root, dirs, files in os.walk('/kaggle/input'):
    dirname_lower = os.path.basename(root).lower()
    if '_32' in dirname_lower or '-32' in dirname_lower:
        continue
    if PARTIAL_RESULTS_NAME in dirname_lower.replace('_', '-'):
        continue

    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            sample_dir = None
            for label in ('real', 'fake'):
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                n_frames = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg')])
                if n_frames > 0 and abs(n_frames - TARGET_T) > TARGET_T * 0.5:
                    print(f'  SKIP: {root} (frames={n_frames}, expected ~{TARGET_T})')
                    continue

            path_lower = root.lower()
            if 'dfd01' in path_lower or 'dfd-01' in path_lower or 'dfd_01' in path_lower:
                ds_name = 'dfd01'
            elif 'dfdc02' in path_lower or 'dfdc-02' in path_lower or 'dfdc' in path_lower:
                ds_name = 'dfdc02'
            elif 'celeb' in path_lower:
                ds_name = 'celebdf'
            else:
                ds_name = os.path.basename(root).replace('-', '_')

            if any(d['name'] == ds_name for d in found_datasets):
                print(f'  SKIP duplicate: {ds_name} at {root}')
                continue

            found_datasets.append({'path': root, 'name': ds_name, 'real': rc, 'fake': fc})
            print(f'  Found: {ds_name} at {root} (real={rc}, fake={fc})')

if len(found_datasets) < 3:
    print(f'WARNING: Expected 3 datasets, found {len(found_datasets)}!')
    print('Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    if len(found_datasets) == 0:
        sys.exit(1)

# Sort datasets alphabetically for consistent naming regardless of os.walk order
found_datasets.sort(key=lambda d: d['name'])

dataset_roots = '+'.join(d['path'] for d in found_datasets)
dataset_names = '+'.join(d['name'] for d in found_datasets)
total_videos = sum(d['real'] + d['fake'] for d in found_datasets)
print(f'\nCombined: {dataset_names}')
print(f'Total videos: {total_videos}')

# =============================================
# Run 4 experiments (with resume skip)
# =============================================
from config import Config
from train import train
from utils import save_metrics

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',    'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

SPLIT_FILENAME = 'split_seed42_3ds_t16.json'

def is_experiment_done(exp, dataset_names):
    """Check if experiment already has metrics.json from a previous run.
    Uses fuzzy matching by model_type to handle dataset name order changes."""
    # Method 1: exact match via Config path
    cfg_tmp = Config()
    cfg_tmp.dataset_name = dataset_names
    cfg_tmp.model_type = exp['model_type']
    cfg_tmp.fusion_type = exp['fusion_type']
    cfg_tmp.num_frames = 16
    cfg_tmp.batch_size = 16
    cfg_tmp.seed = 42
    cfg_tmp.output_dir = './experiments'
    metrics_file = cfg_tmp.metrics_path()
    if os.path.exists(metrics_file):
        try:
            with open(metrics_file) as f:
                data = json.load(f)
            if 'test' in data and data['test'].get('auc', 0) > 0:
                return True, data
        except:
            pass
    # Method 2: fuzzy match — scan all dirs for matching model_type
    exp_dir = './experiments'
    if os.path.isdir(exp_dir):
        mt = exp['model_type']
        for d in os.listdir(exp_dir):
            dp = os.path.join(exp_dir, d)
            if not os.path.isdir(dp):
                continue
            dl = d.lower()
            match = False
            if mt == 'full' and '_full_' in dl and 'adaptive' in dl:
                match = True
            elif mt == 'spatial' and '_spatial_' in dl:
                match = True
            elif mt == 'temporal' and '_temporal_' in dl:
                match = True
            elif mt == 'sequential' and '_sequential_' in dl:
                match = True
            if not match:
                continue
            mf = os.path.join(dp, 'metrics.json')
            if os.path.exists(mf):
                try:
                    with open(mf) as f:
                        data = json.load(f)
                    if 'test' in data and data['test'].get('auc', 0) > 0:
                        print(f'[RESUME] Found completed {mt} in {d}')
                        return True, data
                except:
                    pass
    return False, None

all_results = []

prev_results_path = './experiments/all_results_3ds_t16.json'
if os.path.exists(prev_results_path):
    try:
        with open(prev_results_path) as f:
            all_results = json.load(f)
        print(f'[RESUME] Loaded {len(all_results)} previous results')
    except:
        all_results = []

for i, exp in enumerate(EXPERIMENTS, 1):
    done, prev_metrics = is_experiment_done(exp, dataset_names)
    if done:
        print(f'\n[SKIP] {exp["name"]} already completed — AUC={prev_metrics["test"]["auc"]:.4f}')
        if not any(r.get('experiment_name') == exp['name'] for r in all_results):
            prev_metrics['experiment_name'] = exp['name']
            prev_metrics['status'] = 'success'
            all_results.append(prev_metrics)
        continue

    print(f'\n{"="*70}')
    print(f'[{i}/4] {exp["name"]} (model_type={exp["model_type"]}, T=16, 3-dataset)')
    print(f'{"="*70}\n')

    cfg = Config()
    cfg.dataset_root = dataset_roots
    cfg.dataset_name = dataset_names
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']
    cfg.num_frames = 16
    cfg.max_epochs = 30
    cfg.batch_size = 16
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.splits_dir = './splits'
    cfg.output_dir = './experiments'
    cfg.seed = 42
    cfg.unfreeze_last_n_blocks = 2
    cfg.split_filename = SPLIT_FILENAME

    split_exists = os.path.exists(os.path.join('./splits', SPLIT_FILENAME))
    if split_exists:
        cfg.split_mode = 'fixed'
        cfg.save_split = False
    else:
        cfg.split_mode = 'random'
        cfg.save_split = True

    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f'\n[OK] {exp["name"]} done in {metrics["duration_min"]} min')
        print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
    except Exception as e:
        elapsed = round((time.time() - t0) / 60, 1)
        print(f'\n[FAIL] {exp["name"]} after {elapsed} min: {e}')
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': elapsed,
        })

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

    save_metrics(all_results, './experiments/all_results_3ds_t16.json')

    os.makedirs('/kaggle/working/experiments', exist_ok=True)
    os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
    os.system('cp -r splits/ /kaggle/working/splits/ 2>/dev/null')
    os.system('tar czf /kaggle/working/results_3ds_t16.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
    print(f'[SAVE] Intermediate results saved to /kaggle/working/')

# =============================================
# Summary
# =============================================
print(f'\n{"="*70}')
print('RESULTS SUMMARY (DFDC02+DFD01+CelebDF, T=16)')
print(f'{"="*70}')
print(f'{"Experiment":<25} {"AUC":>8} {"Acc":>8} {"F1":>8} {"EER":>8} {"Epoch":>6} {"Time":>8}')
print('-' * 75)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f'{r["experiment_name"]:<25} {t.get("auc",0):>8.4f} {t.get("accuracy",0):>8.4f} '
              f'{t.get("f1",0):>8.4f} {t.get("eer",0):>8.4f} {be:>6} {r.get("duration_min",0):>7.1f}m')
    else:
        print(f'{r["experiment_name"]:<25} {"FAILED":>8} {r.get("error","")[:40]}')

os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('cp -r splits/ /kaggle/working/splits/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_3ds_t16.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'\nresults_3ds_t16.tar.gz ready for download')
'''

with open('/kaggle/working/run_training_3ds_t16.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training
print('\n=== Starting 3-dataset T=16 training ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_3ds_t16.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'\nTraining subprocess exited with code: {result.returncode}')

if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/results_3ds_t16.tar.gz'):
    print('results_3ds_t16.tar.gz is ready for download')

=== Installing dependencies ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 66.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

Dependencies installed.


...

Project cloned.

=== Starting 3-dataset T=16 training ===


2026-03-22 23:32:11 | INFO | ======================================================================
2026-03-22 23:32:11 | INFO | Эксперимент: celebdf+dfd01+dfdc02_sequential_seed42_bs16_T16
2026-03-22 23:32:11 | INFO | Dataset: celebdf+dfd01+dfdc02
2026-03-22 23:32:11 | INFO | Model: sequential
2026-03-22 23:32:11 | INFO | Fusion: adaptive
2026-03-22 23:32:11 | INFO | Seed: 42
2026-03-22 23:32:11 | INFO | Device: cuda
2026-03-22 23:32:11 | INFO | Batch size: 16
2026-03-22 23:32:11 | INFO | Epochs: 30
2026-03-22 23:32:11 | INFO | Primary metric: auc
2026-03-22 23:32:11 | INFO | ======================================================================
2026-03-22 23:32:11 | INFO | Загрузка данных...
2026-03-22 23:32:20 | INFO | Train: 7001 videos | Val: 1501 | Test: 1506
2026-03-22 23:32:20 | INFO | Создание модели...
2026-03-22 23:32:23 | INFO | Warmup: spatial backbone заморожен (эпохи 1..5)
2026-03-22 23:32:23 | INFO | Параметры: всего=24,669,769, обучаемых=7,121,153
2026-03-22 23:32:23 |

numpy version: 1.26.4
PyTorch: 2.2.2+cu121
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB
[RESUME] Found partial results at /kaggle/input/datasets/alexandertarakanov/3ds-t16-partial/experiments
[RESUME] Restored 3 experiment dirs: ['dfdc02+dfd01+celebdf_spatial_seed42_bs16_T16', 'dfdc02+dfd01+celebdf_full_seed42_bs16_T16_adaptive', 'dfdc02+dfd01+celebdf_temporal_seed42_bs16_T16']
[RESUME] Restored splits from /kaggle/input/datasets/alexandertarakanov/3ds-t16-partial/splits/splits

=== Searching for T=16 datasets ===
  Found: celebdf at /kaggle/input/datasets/alexandertarakanov/celebdf-preprocessed-t16 (real=589, fake=2710)
  Found: dfdc02 at /kaggle/input/datasets/alexandertarakanov/preprocessed-dfdc02-16/preprocessed_DFDC02_16 (real=1727, fake=1566)
  Found: dfd01 at /kaggle/input/datasets/alexandertarakanov/preprocessed-dfd01-16/preprocessed_DFD01_16 (real=363, fake=3068)

Combined: celebdf+dfd01+dfdc02
Total videos: 10023
[RESUME] Loaded 3 previous results
[RESUME] Found completed fu

2026-03-22 23:33:17 | INFO |   [batch 43/437] avg_loss=0.6714
2026-03-22 23:34:08 | INFO |   [batch 86/437] avg_loss=0.6672
2026-03-22 23:35:00 | INFO |   [batch 129/437] avg_loss=0.6687
2026-03-22 23:35:53 | INFO |   [batch 172/437] avg_loss=0.6650
2026-03-22 23:36:46 | INFO |   [batch 215/437] avg_loss=0.6669
2026-03-22 23:37:37 | INFO |   [batch 258/437] avg_loss=0.6661
2026-03-22 23:38:28 | INFO |   [batch 301/437] avg_loss=0.6637
2026-03-22 23:39:19 | INFO |   [batch 344/437] avg_loss=0.6635
2026-03-22 23:40:10 | INFO |   [batch 387/437] avg_loss=0.6628
2026-03-22 23:41:05 | INFO |   [batch 430/437] avg_loss=0.6630
2026-03-22 23:43:30 | INFO | Epoch 01/30 | Train Loss: 0.6627 | Train AUC: 0.5336 | Val Loss: 0.6479 | Val AUC: 0.6143 | Val Acc: 0.6389 | Val F1: 0.7460 | LR backbone: 0.000020 | LR head: 0.000060 | Time: 667.4s
2026-03-22 23:43:30 | INFO | Лучшая модель сохранена: auc = 0.6143
2026-03-22 23:44:04 | INFO |   [batch 43/437] avg_loss=0.6179
2026-03-22 23:44:37 | INFO |  


[OK] A4_sequential done in 197.7 min
     AUC=0.9490  Acc=0.9203  F1=0.9464
GPU Memory after cleanup: 0.02 GB allocated, 0.10 GB reserved
[SAVE] Intermediate results saved to /kaggle/working/

RESULTS SUMMARY (DFDC02+DFD01+CelebDF, T=16)
Experiment                     AUC      Acc       F1      EER  Epoch     Time
---------------------------------------------------------------------------
A1_full_model               0.9587   0.9436   0.9622   0.1211     16   208.3m
A2_spatial_only             0.9494   0.9303   0.9538   0.1479     18   198.3m
A3_temporal_only            0.9539   0.9150   0.9412   0.1168     17   166.5m
A4_sequential               0.9490   0.9203   0.9464   0.1393     20   197.7m

results_3ds_t16.tar.gz ready for download

Training subprocess exited with code: 0
Results copied to /kaggle/working/experiments/
results_3ds_t16.tar.gz is ready for download
